In [1]:
from pathlib import Path  # Dùng để tìm thư mục dự án mà không ghi đường dẫn cố định.
import sys  # Dùng để Python tìm thấy các module trong thư mục src/.

project_root = Path.cwd()  # Lấy thư mục hiện tại của kernel notebook.
if project_root.name == 'notebooks':  # Nếu kernel đang ở notebooks/, lùi lên một cấp.
    project_root = project_root.parent  # Thư mục cha chính là gốc repository.
sys.path.insert(0, str(project_root))  # Thêm gốc dự án vào danh sách nơi Python tìm module.
print('Thư mục dự án:', project_root)  # In ra để người học xác nhận đúng workspace.


Thư mục dự án: c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project



Hai JSONL cần được tạo từ notebook EDA. Cần dùng môi trường Python đã cài các thư viện trong `requirements.txt`, đặc biệt là bản PyTorch có CUDA. Nếu không thấy GPU ở cell dưới, đừng chạy fine-tune vì CPU sẽ chậm hơn rất nhiều.


In [2]:
import torch  # PyTorch kiểm tra GPU và thực hiện phép tính của mô hình.
from src.train_qa import TRAIN_FILE, DEV_FILE  # Lấy đường dẫn dữ liệu mà script sẽ dùng.

print('Train tồn tại:', TRAIN_FILE.exists(), TRAIN_FILE)  # Xác nhận tệp train đã sẵn sàng.
print('Dev tồn tại:', DEV_FILE.exists(), DEV_FILE)  # Xác nhận tệp dev đã sẵn sàng.
print('Có CUDA:', torch.cuda.is_available())  # CUDA cho phép PyTorch dùng GPU NVIDIA.
if torch.cuda.is_available():  # Chỉ hỏi tên thiết bị khi đã phát hiện GPU.
    print('GPU:', torch.cuda.get_device_name(0))  # Kiểm tra notebook nhận đúng card
    print('Hỗ trợ BF16:', torch.cuda.is_bf16_supported())  # BF16 thường tiết kiệm bộ nhớ và ổn định.


c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train tồn tại: True C:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\data\processed\train.json
Dev tồn tại: True C:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\data\processed\dev.json
Có CUDA: True
GPU: NVIDIA GeForce RTX 5070 Ti
Hỗ trợ BF16: True



Cell sau gọi cùng một pipeline dùng cho hai kiến trúc. Với mỗi seed, script tạo mô hình mới từ checkpoint gốc, token hóa dữ liệu, fine-tune ba epoch, đánh giá EM/F1 sau từng epoch và lưu checkpoint tốt nhất theo F1. Không tái sử dụng trọng số của seed trước.

Mỗi lượt sẽ ghi `metrics.json`, `predictions.json` và thư mục `best_model/` dưới `results/finetune/`. Xem đầy đủ logic từng dòng trong `src/train_qa.py`; đặc biệt là `prepare_train_features`, `decode_predictions` và `make_compute_metrics`.


In [4]:
from src.train_qa import run_experiment  # Nạp hàm điều phối tiền xử lý, fine-tune và đánh giá.

seeds = [42, 123, 1024]  # Ba seed theo quy ước của project để đo độ biến động.
model_names = ['mdeberta']  # Notebook này chỉ chạy kiến trúc microsoft/mdeberta-v3-base.
run_experiment(model_names, seeds)  # Chạy lần lượt từng seed và lưu kết quả vào results/.


Đã có kết quả seed 42; giữ nguyên checkpoint đã lưu.

Model: microsoft/mdeberta-v3-base | seed: 123
GPU: NVIDIA GeForce RTX 5070 Ti


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 28208.97it/s]
[transformers] DebertaV2ForQuestionAnswering LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
qa_outputs.wei

Epoch,Training Loss,Validation Loss,Exact Match,F1
1,5.953228,5.950306,12.323020,15.333875
2,5.961401,5.937756,8.652334,11.914064
3,5.938066,5.937500,17.435763,17.597566


Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.22s/it]


Training Loss,Validation Loss,Epoch,Exact Match,F1
5.938066,5.937500,3,17.435763,17.597566


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.10s/it]


... (3 outputs left out)

Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.91s/it]


Training Loss,Validation Loss,Epoch,Exact Match,F1
5.482104,5.940103,3,30.309386,30.311770


Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.21s/it]


Kết quả seed 1024: EM=30.31, F1=30.31

Tổng hợp (F1/EM theo phần trăm):
mdeberta: seed tốt nhất=1024, F1=30.31, EM=30.31; F1 trung bình=21.81 ± 6.01
Đã lưu tổng hợp tại: C:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\finetune\summary.json



`summary.json` ghi seed có F1 cao nhất và điểm trung bình ± độ lệch chuẩn của các seed trong lần chạy. `predictions.json` lưu câu trả lời từng mẫu để xem lỗi. Tên `best_seed` chỉ có nghĩa là seed tốt nhất trong ba seed đã chạy trên tập dev này; nó không đảm bảo sẽ tốt nhất trên tập test hoặc dữ liệu mới.

Khi so sánh XLM-RoBERTa với mDeBERTa, so sánh cả F1/EM trung bình và độ lệch chuẩn, dùng cùng train/dev, seed, epoch, độ dài token và ngưỡng no-answer. Sau khi chọn cấu hình, chỉ dùng test set một lần để báo cáo cuối cùng nếu dự án có test set riêng.


In [5]:
import json  # Đọc tệp kết quả JSON do script tạo ra.
from pathlib import Path  # Tạo đường dẫn kết quả tương đối với gốc repository.

summary_path = project_root / 'results' / 'finetune' / 'mdeberta' / 'summary.json'  # Tổng hợp riêng model này.
if summary_path.exists():  # Chỉ đọc sau khi cell huấn luyện đã chạy xong.
    print(summary_path.read_text(encoding='utf-8'))  # Hiển thị seed tốt nhất, trung bình và độ lệch chuẩn.
else:  # Thí nghiệm có thể chưa chạy hoặc chưa kết thúc.
    print('Chưa có summary; hãy chạy cell huấn luyện đến khi hoàn tất.')  # Thông báo bước kế tiếp.


{
  "best_seed": 1024,
  "best_f1": 30.311770033846592,
  "best_exact_match": 30.309386470896698,
  "mean_f1": 21.81390803574276,
  "std_f1": 6.0089547576341396,
  "mean_exact_match": 21.630833770319878,
  "std_exact_match": 6.13779292447539
}
